Este notebook debe conectarse localmente utilizando el URL obtenido al ejecutar las siguientes líneas en la terminal:
```
cd ~/py4-docker
docker compose logs jupyter 2>&1 | grep "127.0.0.1:8888"
```
Haz clic en la esquina superior derecha:  
Connect → Connect to local runtime...   
Pega el URL ("http://127.0.0.1:8888/?token=...") y conecta.


# Persistencia en MySQL con PySpark y JDBC

En esta sesión vamos a completar un pipeline de Big Data utilizando PySpark, JDBC y MySQL dentro de un entorno Docker.

El flujo será:

**CSV → PySpark → procesamiento → DataFrame → JDBC → MySQL → DataFrame**

En este ejercicio trabajaremos con dos containers:

* **Jupyter:** Python, PySpark y Java.
* **MySQL:** servidor de base de datos.

Google Colab se utiliza como interfaz para ejecutar el notebook, pero el código se ejecuta en el runtime local de Jupyter dentro de Docker.


## Objetivos

Al finalizar este ejercicio podremos:

1. Configurar Spark para utilizar el driver JDBC de MySQL.
2. Procesar nuevamente el dataset de transacciones.
3. Persistir un `DataFrame` de Spark directamente en MySQL.
4. Leer la tabla nuevamente desde MySQL usando Spark.
5. Verificar que los datos realmente quedaron almacenados en la base de datos.

El objetivo principal es observar que Spark puede trabajar directamente con una base de datos mediante JDBC, sin convertir el `DataFrame` a Pandas.


## Arquitectura del entorno Docker

En este ejercicio, los componentes principales están organizados de la siguiente manera:

```text
Google Colab
     ↓
Jupyter container
     ↓
PySpark + Java
     ↓
JDBC
     ↓
MySQL container
     ↓
bigdata_py4
```




## 1. Configuración de Spark

Primero crearemos la aplicación Spark.

El driver JDBC de MySQL ya está incluido dentro de la imagen Docker en:

```text
/opt/jdbc/mysql-connector-j-26.7.0.jar
```

Configuraremos el `classpath` de Java para que Spark pueda encontrar este driver.

Es importante realizar esta configuración cuando se crea la `SparkSession`, antes de iniciar el trabajo con JDBC.


In [1]:
from pyspark.sql import SparkSession  # Importamos SparkSession para crear la aplicación Spark.

spark = (
    SparkSession.builder  # Iniciamos la configuración de nuestra SparkSession.
    .appName("PY4_BigData_Docker")  # Definimos el nombre de la aplicación Spark.
    .master("local[2]")  # Ejecutamos Spark localmente utilizando 2 cores.
    .config(  # Configuramos el classpath del Spark driver.
        "spark.driver.extraClassPath",
        "/opt/jdbc/mysql-connector-j-26.7.0.jar"  # Indicamos la ubicación del JDBC Driver de MySQL.
    )
    .config(  # Configuramos el classpath de los Spark executors.
        "spark.executor.extraClassPath",
        "/opt/jdbc/mysql-connector-j-26.7.0.jar"  # Indicamos la ubicación del JDBC Driver para los executors.
    )
    .getOrCreate()  # Creamos la SparkSession utilizando la configuración anterior.
)

print("Spark version:", spark.version)  # Mostramos la versión de Spark que está ejecutándose.

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/09 20:19:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.0.4


### Configuración de la conexión JDBC

Ahora definiremos los parámetros necesarios para conectarnos a MySQL.

Utilizaremos `getpass` para solicitar el password sin mostrarlo directamente en el notebook.

La URL JDBC utilizará:

```text
jdbc:mysql://mysql:3306/bigdata_py4
```

`mysql` es el nombre del servicio dentro de Docker y `3306` es el puerto interno de MySQL.


In [2]:
from getpass import getpass  # Importamos una función para pedir el password sin mostrarlo.

MYSQL_PASSWORD = getpass(
    "MySQL password: "
)  # Pedimos el password de MySQL de forma segura.

jdbc_url = "jdbc:mysql://mysql:3306/bigdata_py4"  # Definimos la URL JDBC para conectarnos al servicio MySQL dentro de Docker.

jdbc_properties = {  # Definimos las propiedades necesarias para que Spark pueda conectarse a MySQL mediante JDBC.
    "user": "root", # ek usuario es "root"
    "password": MYSQL_PASSWORD, # Utilizamos el password introducido anteriormente sin escribirlo directamente en el notebook.
    "driver": "com.mysql.cj.jdbc.Driver" # Indicamos la clase Java correspondiente al JDBC Driver de MySQL.
}  # Definimos las propiedades necesarias para la conexión JDBC.

MySQL password: ··········


El password es: "MYSQL_PASSWORD"

### Prueba de conexión

Antes de procesar el dataset completo, ejecutaremos una consulta pequeña.

La consulta:

```sql
SELECT 1
```

nos permite comprobar que:

* Spark puede utilizar el driver JDBC.
* Spark puede resolver el servicio `mysql`.
* MySQL está disponible.
* Las credenciales son correctas.
* La base de datos `bigdata_py4` está accesible.


In [3]:
# Ejecutamos la lectura y creamos un DataFrame de Spark con el resultado.

df_test = (
    spark.read  # Iniciamos una operación de lectura utilizando Spark.
    .format("jdbc")  # Indicamos que los datos se leerán mediante una conexión JDBC.
    .option("url", jdbc_url)  # Indicamos la URL JDBC que define dónde está nuestra base de datos MySQL.
    .option(
        "dbtable",
        "(SELECT 1 AS prueba) AS t"  # Ejecutamos una consulta SQL pequeña para comprobar que la conexión funciona.
                                     # "SELECT 1" devuelve un único valor, por lo que no necesitamos ninguna tabla existente.
                                     # "AS prueba" asigna el nombre "prueba" al resultado de la consulta.
                                     # "AS t" crea un alias para que Spark pueda utilizar la consulta como una tabla.
    )
    .options(**jdbc_properties)  # Pasamos a Spark el usuario, password y JDBC Driver definidos anteriormente.
    .load()  # Ejecutamos la lectura y creamos un DataFrame de Spark con el resultado.
)


df_test.show()  # Mostramos el resultado de la prueba.

+------+
|prueba|
+------+
|     1|
+------+



## 2. Procesamiento del dataset

Ahora volveremos al dataset de transacciones utilizado en el ejercicio anterior.

Primero definiremos explícitamente el `schema`.

Esto nos permite controlar los tipos de datos que Spark utilizará al leer el CSV y evitar inferencias innecesarias.


In [4]:
from pyspark.sql import functions as F  # Importamos las funciones de Spark SQL.

from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType,
    TimestampType
)  # Importamos los tipos necesarios para definir el schema.

schema_transacciones = StructType([
    StructField("id_transaccion", IntegerType(), True),
    StructField("cliente_id", IntegerType(), True),
    StructField("categoria", StringType(), True),
    StructField("monto", DoubleType(), True),
    StructField("canal", StringType(), True),
    StructField("fecha_hora", TimestampType(), True),
])  # Definimos manualmente el schema para controlar los tipos de datos.

ruta_csv = "data/transacciones_masivas.csv"  # Definimos la ruta del dataset.

### Lectura del dataset

Leeremos el archivo CSV directamente con Spark.

No utilizaremos Pandas, ya que el objetivo es mantener el procesamiento dentro del ecosistema Spark.


In [5]:
df_spark = spark.read.csv(
    ruta_csv,
    header=True,
    schema=schema_transacciones
)  # Leemos el CSV aplicando el schema definido.

df_spark.printSchema()  # Verificamos los tipos de datos.

df_spark.show(5)  # Mostramos algunas filas para comprobar la lectura.

root
 |-- id_transaccion: integer (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- categoria: string (nullable = true)
 |-- monto: double (nullable = true)
 |-- canal: string (nullable = true)
 |-- fecha_hora: timestamp (nullable = true)

+--------------+----------+-----------+------+-----+-------------------+
|id_transaccion|cliente_id|  categoria| monto|canal|         fecha_hora|
+--------------+----------+-----------+------+-----+-------------------+
|          4680|      2811|      hogar| 10.07|  Web|2024-01-15 23:51:00|
|         42232|      1055|      Hogar| 72.17|  App|2024-02-21 22:22:00|
|         39843|      4162|   Deportes|   6.3|  Web|2024-01-06 00:27:00|
|         27758|      2322|Electronica| 76.01|  Web|2024-02-14 03:47:00|
|         29348|      4966|  ALIMENTOS|113.44|  Web|2024-02-12 11:56:00|
+--------------+----------+-----------+------+-----+-------------------+
only showing top 5 rows


### Limpieza de datos

Aplicaremos varias transformaciones para preparar el dataset:

* eliminar transacciones duplicadas;
* eliminar registros sin `monto`;
* eliminar registros sin `categoria`;
* eliminar montos menores o iguales a cero;
* estandarizar el nombre de las categorías.

El resultado será un nuevo `DataFrame` llamado `df_limpio`.


In [6]:
df_limpio = df_spark \
    .dropDuplicates(["id_transaccion"]) \
    .dropna(subset=["monto", "categoria"]) \
    .filter(F.col("monto") > 0) \
    .withColumn(
        "categoria",
        F.initcap(F.lower(F.col("categoria")))
    )  # Limpiamos duplicados, valores nulos, montos inválidos y estandarizamos categoria.

df_limpio.show(5)  # Mostramos algunas filas después de la limpieza.

print(
    "Filas después de la limpieza:",
    df_limpio.count()
)  # Contamos las filas resultantes.

+--------------+----------+----------+------+-----+-------------------+
|id_transaccion|cliente_id| categoria| monto|canal|         fecha_hora|
+--------------+----------+----------+------+-----+-------------------+
|             1|      2844|Tecnologia| 39.14|  Web|2024-02-29 22:39:00|
|             3|      2200|      Ropa|247.17|  Web|2024-01-21 05:02:00|
|             5|      2543|      Ropa| 69.54|  Web|2024-02-04 18:26:00|
|             6|      2190| Alimentos| 12.65|  App|2024-01-18 10:13:00|
|             9|      4810|Tecnologia| 100.3|  App|2024-02-23 19:23:00|
+--------------+----------+----------+------+-----+-------------------+
only showing top 5 rows


Filas después de la limpieza: 59157


### Transformación de los datos

Calcularemos una nueva columna llamada `total_con_impuestos`.

Para este ejercicio utilizaremos una tasa de impuesto del 13%.

También extraeremos la hora de la transacción a partir de `fecha_hora`.


In [7]:
IMPUESTO = 0.13  # Definimos la tasa de impuesto.

df_procesado = df_limpio \
    .withColumn(
        "total_con_impuestos",
        F.round(F.col("monto") * (1 + IMPUESTO), 2)
    ) \
    .withColumn(
        "hora",
        F.hour(F.col("fecha_hora"))
    )  # Calculamos el total con impuestos y extraemos la hora.

df_procesado.select(
    "id_transaccion",
    "categoria",
    "monto",
    "total_con_impuestos",
    "hora"
).show(5)  # Verificamos las nuevas columnas.

+--------------+----------+------+-------------------+----+
|id_transaccion| categoria| monto|total_con_impuestos|hora|
+--------------+----------+------+-------------------+----+
|             1|Tecnologia| 39.14|              44.23|  22|
|             2|  Deportes|114.84|             129.77|  17|
|             3|      Ropa|247.17|              279.3|   5|
|             4| Alimentos| 49.58|              56.03|   3|
|             5|      Ropa| 69.54|              78.58|  18|
+--------------+----------+------+-------------------+----+
only showing top 5 rows


### Agregación por categoría

Finalmente agruparemos las transacciones por `categoria`.

Para cada categoría calcularemos:

* el promedio del total con impuestos;
* la cantidad de transacciones.

Este será el `DataFrame` que posteriormente persistiremos en MySQL.


In [8]:
resumen_por_categoria = df_procesado \
    .groupBy("categoria") \
    .agg(
        F.avg("total_con_impuestos").alias("promedio_total"),
        F.count("*").alias("cantidad_transacciones")
    ) \
    .orderBy(
        F.col("promedio_total").desc()
    )  # Agrupamos por categoria, calculamos métricas y ordenamos por promedio.

resumen_por_categoria.show()  # Mostramos el resumen final.

+-----------+------------------+----------------------+
|  categoria|    promedio_total|cantidad_transacciones|
+-----------+------------------+----------------------+
|  Alimentos|171.89365582628207|                  9809|
|       Ropa| 170.2257657114126|                  9945|
|   Deportes|166.07061272504083|                  9776|
| Tecnologia|165.40620332159003|                  9935|
|      Hogar|164.54979538087608|                  9872|
|Electronica| 163.9283085539719|                  9820|
+-----------+------------------+----------------------+



## 3. Persistencia en MySQL

Ahora realizaremos el paso principal del ejercicio.

Escribiremos directamente el `DataFrame` de Spark en una tabla MySQL utilizando JDBC.

No convertiremos el `DataFrame` a Pandas.

Spark proporciona una Data Source JDBC que permite escribir un `DataFrame` directamente en una tabla externa.

Utilizaremos `overwrite` para que cada ejecución del notebook reconstruya la tabla con el resultado actual.


In [10]:
tabla_mysql = "resumen_ventas_categoria"   # Definimos el nombre de la tabla que crearemos en la base de datos MySQL.

# Persistimos el DataFrame directamente en MySQL mediante JDBC:
(
    resumen_por_categoria.write # Iniciamos la operación para escribir el DataFrame de Spark.
    .format("jdbc") #  Utilizaremos JDBC para enviar los datos a MySQL
    .option("url", jdbc_url) # Indicamos la URL JDBC
    .option("dbtable", tabla_mysql) # Indicamos el nombre de la tabla MySQL donde almacenaremos el DataFrame.
    .options(**jdbc_properties) # Pasamos las propiedades de conexión: usuario, password y JDBC Driver.
    .mode("overwrite") # Indicamos que Spark debe reemplazar la tabla si ya existe.
    .save() # Ejecutamos la escritura y persistimos los datos del DataFrame en MySQL.
)

print(
    f"Tabla '{tabla_mysql}' almacenada correctamente en MySQL."
)  # Confirmamos que terminó la escritura.

Tabla 'resumen_ventas_categoria' almacenada correctamente en MySQL.


## 4. Lectura de la tabla desde MySQL

Ahora realizaremos el proceso inverso.

Spark leerá nuevamente la tabla MySQL y construirá un nuevo `DataFrame`.

De esta forma podemos comprobar que los datos no solamente fueron enviados a MySQL, sino que también pueden recuperarse nuevamente mediante JDBC.


In [11]:
# Leemos la tabla MySQL utilizando Spark y JDBC.
resumen_mysql = (
    spark.read # Iniciamos una operación de lectura utilizando Spark.
    .format("jdbc") # Indicamos que los datos se leerán desde MySQL mediante JDBC.
    .option("url", jdbc_url) # Indicamos la URL JDBC que identifica el servidor y la base de datos MySQL.
    .option("dbtable", tabla_mysql) # Indicamos la tabla MySQL que queremos leer.
    .options(**jdbc_properties) # Pasamos las propiedades necesarias para conectarnos: usuario, password y JDBC Driver
    .load() # Ejecutamos la lectura y cargamos los datos de MySQL en un DataFrame de Spark.
)

resumen_mysql.show()  # Mostramos los datos recuperados desde MySQL.

+-----------+------------------+----------------------+
|  categoria|    promedio_total|cantidad_transacciones|
+-----------+------------------+----------------------+
|  Alimentos|171.89365582628207|                  9809|
|       Ropa| 170.2257657114126|                  9945|
|   Deportes|166.07061272504083|                  9776|
| Tecnologia|165.40620332159003|                  9935|
|      Hogar|164.54979538087608|                  9872|
|Electronica| 163.9283085539719|                  9820|
+-----------+------------------+----------------------+



## Verificación directa en MySQL

Como última comprobación, podemos consultar directamente el servidor MySQL desde el Terminal.

Esta consulta se ejecutará desde el container de MySQL y no desde Spark.

Así podemos comprobar que la tabla existe realmente en la base de datos y que contiene los registros almacenados por Spark.


En el terminal:
```
docker compose exec mysql mysql -u root -p bigdata_py4
```

Enter password.  

Despues:  

```
SHOW TABLES;

SELECT *
FROM resumen_ventas_categoria;

DESCRIBE resumen_ventas_categoria;

exit;

```

## Inserta datos con sentencias preparadas

**--> Una sentencia preparada (PreparedStatement) permite ejecutar SQL de forma más segura y estructurada, separando la consulta de los valores que se insertan.**





Utilizaremos una **sentencia preparada (`PreparedStatement`)** para insertar un registro de prueba y, posteriormente, ejecutaremos un `SELECT` para recuperar ese registro desde MySQL.

De esta manera, comprobamos que la información realmente quedó almacenada en la base de datos y que podemos recuperarla mediante una conexión JDBC. Spark también utiliza JDBC para sus operaciones de lectura y escritura con bases de datos externas.

In [ ]:
# Obtenemos una conexión JDBC desde Java utilizando el JDBC Driver de MySQL.
connection = spark._sc._gateway.jvm.java.sql.DriverManager.getConnection(
    jdbc_url,  # Indicamos la URL de conexión a nuestra base de datos MySQL.
    jdbc_properties["user"],  # Indicamos el usuario de MySQL.
    jdbc_properties["password"]  # Indicamos el password de MySQL.
)

# Definimos una sentencia SQL parametrizada para insertar una fila de prueba.
sql_insert = """
INSERT INTO resumen_ventas_categoria
(categoria, promedio_total, cantidad_transacciones)
VALUES (?, ?, ?)
"""


Al lugar de escribir directamente los valores asi:
```
VALUES ('Prueba', 100.00, 1)
```
Hace mos un prepared statement:
```
VALUES (?, ?, ?)
```

In [12]:
# Creamos un PreparedStatement utilizando la sentencia SQL parametrizada.
prepared_statement = connection.prepareStatement(sql_insert)

# Asignamos los valores a los parámetros de la sentencia SQL.
prepared_statement.setString(1, "Prueba")  # Asignamos la categoría de prueba.
prepared_statement.setDouble(2, 100.00)  # Asignamos un promedio de prueba.
prepared_statement.setInt(3, 1)  # Asignamos una cantidad de transacciones de prueba.

# Ejecutamos el INSERT en MySQL.
prepared_statement.executeUpdate()

# Cerramos el PreparedStatement porque ya no lo necesitamos.
prepared_statement.close()

In [13]:
# Creamos una sentencia SQL para consultar la fila que acabamos de insertar.
sql_select = """
SELECT categoria, promedio_total, cantidad_transacciones
FROM resumen_ventas_categoria
WHERE categoria = 'Prueba'
"""

# Creamos un Statement para ejecutar la consulta SQL.
statement = connection.createStatement()

# Ejecutamos el SELECT y guardamos el resultado.
result_set = statement.executeQuery(sql_select)

# Recorremos los resultados obtenidos desde MySQL.
while result_set.next():
    print(
        "Categoria:", result_set.getString("categoria"),
        "| Promedio:", result_set.getDouble("promedio_total"),
        "| Transacciones:", result_set.getInt("cantidad_transacciones")
    )

# Cerramos los recursos JDBC utilizados para la consulta.
result_set.close()
statement.close()
connection.close()

Categoria: Prueba | Promedio: 100.0 | Transacciones: 1


# Conclusiones

En este ejercicio completamos un pipeline de persistencia utilizando PySpark, JDBC y MySQL.

El flujo completo fue:

```text
CSV
 ↓
PySpark
 ↓
DataFrame
 ↓
Transformaciones
 ↓
Agregación
 ↓
JDBC
 ↓
MySQL
 ↓
JDBC
 ↓
DataFrame
```

Los puntos principales fueron:

* Spark utilizó el driver JDBC de MySQL.
* El dataset fue procesado completamente con Spark.
* El `DataFrame` agregado fue persistido directamente en MySQL.
* La tabla fue leída nuevamente utilizando Spark.
* Finalmente verificamos los datos directamente en MySQL.

Este flujo permite integrar procesamiento distribuido con sistemas tradicionales de almacenamiento relacional sin convertir el `DataFrame` de Spark a Pandas.
